# 01. 1차 미분방정식과 수치해

미분방정식은 상태의 변화율이 현재 상태와 입력에 의해 결정되는 모델이다.

$$\dot{x}=f(x,t)$$

로봇 동역학, 모터 응답, 배터리/필터 모델, 제어기는 대부분 이 형태에서 시작한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

## 1. Euler vs RK4

1차 시스템:

$$\dot{x}=-kx+u$$

해석해와 수치해를 비교한다.

In [ ]:
k = 1.4
u = 2.0
x0 = 0.0
T = 5.0
dt = 0.08
t = np.arange(0, T + dt, dt)

def f(x, time):
    return -k*x + u

def euler_step(x, time, dt):
    return x + dt * f(x, time)

def rk4_step(x, time, dt):
    k1 = f(x, time)
    k2 = f(x + 0.5*dt*k1, time + 0.5*dt)
    k3 = f(x + 0.5*dt*k2, time + 0.5*dt)
    k4 = f(x + dt*k3, time + dt)
    return x + dt * (k1 + 2*k2 + 2*k3 + k4) / 6

x_eu = [x0]
x_rk = [x0]
for i in range(len(t)-1):
    x_eu.append(euler_step(x_eu[-1], t[i], dt))
    x_rk.append(rk4_step(x_rk[-1], t[i], dt))
x_eu = np.array(x_eu); x_rk = np.array(x_rk)
x_true = (x0 - u/k) * np.exp(-k*t) + u/k

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(t, x_true, 'k-', lw=2.5, label='analytical')
axes[0].plot(t, x_eu, 'o-', color='#E85D24', ms=3, label='Euler')
axes[0].plot(t, x_rk, '--', color='#1D9E75', lw=2, label='RK4')
axes[0].grid(alpha=0.25); axes[0].legend(); axes[0].set_title('1차 시스템 응답')

axes[1].semilogy(t, np.abs(x_eu - x_true) + 1e-12, color='#E85D24', label='Euler error')
axes[1].semilogy(t, np.abs(x_rk - x_true) + 1e-12, color='#1D9E75', label='RK4 error')
axes[1].grid(alpha=0.25); axes[1].legend(); axes[1].set_title('수치해 오차')
plt.tight_layout()
plt.savefig('assets/01_euler_rk4.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. 모터 속도 응답 모델

DC 모터 속도를 단순화하면 1차 지연 시스템으로 볼 수 있다.

$$\dot{\omega}=\frac{1}{\tau}(K u - \omega)$$

시간상수 $\tau$ 가 작을수록 목표 속도에 빨리 도달한다.

In [ ]:
def simulate_motor(tau, K=10.0, u=1.0, dt=0.01, T=3.0):
    ts = np.arange(0, T + dt, dt)
    w = np.zeros_like(ts)
    for k in range(len(ts)-1):
        wdot = (K*u - w[k]) / tau
        w[k+1] = w[k] + dt * wdot
    return ts, w

fig, ax = plt.subplots(figsize=(8, 5))
for tau, color in [(0.15, '#1D9E75'), (0.4, '#534AB7'), (0.9, '#E85D24')]:
    ts, w = simulate_motor(tau)
    ax.plot(ts, w, lw=2.5, color=color, label=f'tau={tau}s')
ax.axhline(10, color='gray', linestyle='--', lw=1.5, label='steady speed')
ax.set_xlabel('time (s)'); ax.set_ylabel('wheel speed')
ax.set_title('모터 1차 응답')
ax.grid(alpha=0.25); ax.legend()
plt.savefig('assets/01_motor_response.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 안정성 직관

선형 1차 시스템 $\dot{x}=ax$ 에서 $a<0$ 이면 원점으로 수렴하고, $a>0$ 이면 발산한다.

In [ ]:
t = np.linspace(0, 4, 300)
fig, ax = plt.subplots(figsize=(8, 5))
for a, color in [(-2.0, '#1D9E75'), (-0.6, '#534AB7'), (0.4, '#E85D24')]:
    x = np.exp(a*t)
    ax.plot(t, x, lw=2.5, color=color, label=f'a={a}')
ax.set_ylim(0, 5)
ax.set_xlabel('time'); ax.set_ylabel('x(t)')
ax.set_title('xdot = a x 의 안정성')
ax.grid(alpha=0.25); ax.legend()
plt.savefig('assets/01_stability.png', dpi=150, bbox_inches='tight')
plt.show()

## 요약

| 개념 | 의미 | 로보틱스 활용 |
|------|------|---------------|
| ODE | 변화율 모델 | 로봇 동역학, 센서 필터 |
| Euler | 가장 단순한 적분기 | 실시간 근사, 작은 dt 필요 |
| RK4 | 고정밀 수치 적분 | 시뮬레이션 검증 |
| 안정성 | 시간이 지나며 수렴/발산 | 제어기 설계의 핵심 |